# Alkaliphile-secretome fine-tuning — full GPU notebook (Noam recipe)
Published recipe = **Noam warmup**. Runs the whole suite on one T4 and **saves everything to your Google Drive**
(`MyDrive/alkaline_finetuning/`), so you can start it and walk away. **Do the setup at the top (GPU, Drive auth,
upload), then Run all and leave it.** ~50 min.


## Setup — do these first, then leave it running


In [ ]:
# 1. GPU check
import torch, subprocess
print(subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True).stdout.strip() or "NO GPU")
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU, then rerun." 


In [ ]:
# 2. Dependencies (Colab lacks biotite; biopython provides Bio)
!pip -q install biotite biopython
print("deps ready")


In [ ]:
# 3. Mount Google Drive and route ALL outputs there (via symlink -> survives disconnects)
from google.colab import drive; drive.mount('/content/drive')
import os, shutil
DRIVE="/content/drive/MyDrive/alkaline_finetuning"
os.makedirs(DRIVE+"/outputs/evaluation",exist_ok=True); os.makedirs(DRIVE+"/figures",exist_ok=True)
for local,target in [("/content/outputs",DRIVE+"/outputs"),("/content/figures",DRIVE+"/figures")]:
    if not os.path.islink(local):
        if os.path.exists(local): shutil.rmtree(local)
        os.symlink(target,local)
print("outputs + figures -> ",DRIVE)


In [ ]:
# 4. Clone ProteinMPNN
import os, subprocess
if not os.path.exists("/content/ProteinMPNN"):
    subprocess.run(["git","clone","--depth","1","https://github.com/dauparas/ProteinMPNN.git","/content/ProteinMPNN"],check=True)
print("ready")


In [ ]:
# 5. Upload the bundle(s) — pick alkaline_colab_all.zip, OR select BOTH alkaline_colab_upload.zip + alkaline_colab_eval_inputs.zip together.
#    (multi-select in the dialog = one interaction, then you're done uploading)
import zipfile, os
from google.colab import files
up=files.upload()
for name in up:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall('/content'); print("extracted",name)
os.makedirs('/content/outputs/evaluation',exist_ok=True)
print("structures:", sum(len(fs) for _,_,fs in os.walk('/content/data/structures_alkaliphile')))


## 1. Main result — Noam recipe (published)
Train FT_alk + symmetric FT_neu with Noam warmup (save every 2). Select on **validation** by *max steer within the
−3pp recovery guardrail*; run the locked **test** eval at that matched epoch.


In [ ]:
# 1a. Train FT_alk + FT_neu (Noam warmup, save every 2 epochs)
!python /content/train.py --mpnn /content/ProteinMPNN --run alkaliphile_v1 --role case    --jsonl_dir /content --out_root /content/outputs --epochs 30 --save_every 2 --schedule noam
!python /content/train.py --mpnn /content/ProteinMPNN --run neutralophile_control_v1 --role control --jsonl_dir /content --out_root /content/outputs --epochs 30 --save_every 2 --schedule noam


In [ ]:
# 1b. Select on VALIDATION: max steer within the recovery guardrail -> writes chosen epoch + dial CSV (fig5)
!python /content/alkmpnn/sweeps.py --mode checkpoint --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt \
    --ckpt_dir /content/outputs/alkaliphile_v1/model_weights --split val --select maxsteer \
    --select_out /content/outputs/evaluation/selected_epoch.txt --save_csv /content/outputs/evaluation/main_checkpoint_sweep.csv


In [ ]:
# 1c. Locked TEST eval at the selected (matched) epoch -> verdict + surface-vs-core + collapse + multi-pH
ep=open("/content/outputs/evaluation/selected_epoch.txt").read().strip(); print("selected epoch:", ep)
!python /content/alkmpnn/evaluate.py --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt \
    --ft_alk /content/outputs/alkaliphile_v1/model_weights/epoch_{ep}.pt \
    --ft_neu /content/outputs/neutralophile_control_v1/model_weights/epoch_{ep}.pt --n 8
print(open("/content/outputs/evaluation/verdict.txt").read())


## 2. Robustness (steelman)


In [ ]:
# Speed companion: CONSTANT-lr saturates in ~1 epoch (the contrast that motivates checkpoint selection)
!python /content/train.py --mpnn /content/ProteinMPNN --run alkaliphile_const_v1 --role case --jsonl_dir /content --out_root /content/outputs --epochs 30 --save_every 3
!python /content/alkmpnn/sweeps.py --mode checkpoint --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt \
    --ckpt_dir /content/outputs/alkaliphile_const_v1/model_weights --split val --save_csv /content/outputs/evaluation/const_checkpoint_sweep.csv


In [ ]:
# R2. Seed stability (fast constant-lr probe, 5 seeds x 1 epoch)
for s in range(5):
    !python /content/train.py --mpnn /content/ProteinMPNN --run alkaliphile_seed{s} --role case --jsonl_dir /content --out_root /content/outputs --epochs 1 --save_every 1 --seed {s}
ck=",".join(f"/content/outputs/alkaliphile_seed{s}/model_weights/epoch_00.pt" for s in range(5))
!python /content/alkmpnn/sweeps.py --mode seeds --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt --ckpts {ck} --split val


In [ ]:
# R3. Temperature sweep on the SELECTED published epoch (honest magnitude at T=1.0)
ep=open("/content/outputs/evaluation/selected_epoch.txt").read().strip()
!python /content/alkmpnn/sweeps.py --mode temperature --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt \
    --ckpt /content/outputs/alkaliphile_v1/model_weights/epoch_{ep}.pt --save_csv /content/outputs/evaluation/temperature_sweep.csv


In [ ]:
# R4. Readout-only probe: only W_out trainable (0.16% of params)
!python /content/train.py --mpnn /content/ProteinMPNN --run alkaliphile_freeze_v1 --role case --jsonl_dir /content --out_root /content/outputs --epochs 15 --save_every 1 --freeze wout
!python /content/alkmpnn/sweeps.py --mode checkpoint --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt \
    --ckpt_dir /content/outputs/alkaliphile_freeze_v1/model_weights --split val --save_csv /content/outputs/evaluation/freeze_checkpoint_sweep.csv


## 3. Additional experiments
Items 1/3/9 (surface-vs-core, collapse, multi-pH) are in the §1c verdict + fig7. Below: logit-bias baseline (6),
negative control (4), leave-one-clade-out (5), dose-response (8) — fast constant-lr probes.


In [ ]:
# 6. Baseline: hand-coded logit bias toward acidic on BASE (no training)
!python /content/alkmpnn/sweeps.py --mode logitbias --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt --split val --biases "0,0.5,1,2,3,4" --save_csv /content/outputs/evaluation/logitbias_sweep.csv


In [ ]:
# 5. Leave-one-clade-out: drop the dominant clade (Bacilli, 174/252); do the other lineages still steer?
!python /content/train.py --mpnn /content/ProteinMPNN --run alkaliphile_noBacilli --role case --jsonl_dir /content --out_root /content/outputs --epochs 1 --save_every 1 --exclude_clade Bacilli
!python /content/alkmpnn/sweeps.py --mode checkpoint --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt --ckpt_dir /content/outputs/alkaliphile_noBacilli/model_weights --split val --min_gap 0.0


In [ ]:
# 8. Dose-response: 25/50/100% of cases (1 epoch each)
for fr,tag in [(0.25,"25"),(0.5,"50"),(1.0,"100")]:
    !python /content/train.py --mpnn /content/ProteinMPNN --run alk_frac{tag} --role case --jsonl_dir /content --out_root /content/outputs --epochs 1 --save_every 1 --frac {fr}
ck=",".join(f"/content/outputs/alk_frac{t}/model_weights/epoch_00.pt" for t in ["25","50","100"])
!python /content/alkmpnn/sweeps.py --mode seeds --mpnn /content/ProteinMPNN --base /content/ProteinMPNN/vanilla_model_weights/v_48_002.pt --ckpts {ck} --split val


## 4. Figures (also saved to Drive)


In [ ]:
import os
!python /content/alkmpnn/figures.py --checkpoint_csv /content/outputs/evaluation/main_checkpoint_sweep.csv --temperature_csv /content/outputs/evaluation/temperature_sweep.csv
from IPython.display import Image, display
for f in ["fig1_training_curves","fig2_steering_bars","fig3_paired_surface_net","fig4_recovery_guardrail","fig5_steering_curve","fig6_temperature_curve","fig7_surface_vs_core"]:
    p=f"/content/figures/{f}.png"
    if os.path.exists(p): print(f); display(Image(p))


In [ ]:
# Everything is already in Drive (symlinked). Summary:
import os
DRIVE="/content/drive/MyDrive/alkaline_finetuning"
print("DONE. All outputs in", DRIVE)
print("verdict:", os.path.exists(DRIVE+"/outputs/evaluation/verdict.txt"))
print("figures:", sorted(f for f in os.listdir(DRIVE+"/figures") if f.endswith(".png")))
print("sweep CSVs:", sorted(f for f in os.listdir(DRIVE+"/outputs/evaluation") if f.endswith(".csv")))


## 5. Optional: structural plausibility (ESMFold) — heavy ~3 GB, UNTESTED cell


In [ ]:
!pip -q install fair-esm
import torch, esm, numpy as np, sys
m=esm.pretrained.esmfold_v1().eval().cuda()
sys.path.insert(0,"/content/alkmpnn"); sys.path.insert(0,"/content/ProteinMPNN"); import utils as U
dev=U.pick_device(); ep=open("/content/outputs/evaluation/selected_epoch.txt").read().strip()
base=U.load_model("/content/ProteinMPNN/vanilla_model_weights/v_48_002.pt",dev); ft=U.load_model("/content/outputs/alkaliphile_v1/model_weights/epoch_"+ep+".pt",dev)
bb=[b for b in U.collect_backbones("test") if b[1]=="neutralophile"][:5]
def plddt(seq):
    with torch.no_grad(): pdb=m.infer_pdb(seq)
    return np.mean([float(l[60:66]) for l in pdb.split(chr(10)) if l.startswith("ATOM") and l[12:16].strip()=="CA"])
for acc,grp,pdb in bb:
    sb=U.design_backbone(base,pdb,1,0.1,dev)[0][0]; sf=U.design_backbone(ft,pdb,1,0.1,dev)[0][0]
    print(f"{acc}: base pLDDT {plddt(sb):.1f} | FT pLDDT {plddt(sf):.1f}")
